NER MODEL

MedQuAD\
Source 1: https://github.com/abachaa/MedQuAD/tree/master/10_MPlus_ADAM_QA\
Source 2:https://www.kaggle.com/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research

In [5]:
import xml.etree.ElementTree as ET
import glob
import os

def combine_medical_questions(input_dir, output_file="./datasets/combined_medical_questions.xml"):
    # 1. Create the container root element
    combined_root = ET.Element('CombinedMedicalQuestions')

    # Get all XML files in the specified directory
    xml_files = sorted(glob.glob(os.path.join(input_dir, '*.xml')))
    
    if not xml_files:
        print(f"No XML files found in the directory: {input_dir}")
        return
    
    print(f"Found {len(xml_files)} XML files. Processing...")

    for xml_file in xml_files:
        try:
            tree = ET.parse(xml_file)
            # Use a different variable name (current_file_root) to avoid overwriting combined_root
            current_file_root = tree.getroot()
            
            # 2. Extract every Question tag and append it directly
            for q_tag in current_file_root.findall(".//Question"):
                # We append the tag as is (keeping only qid and qtype)
                combined_root.append(q_tag)
                
        except Exception as e:
            print(f"Skipping {os.path.basename(xml_file)} due to error: {e}")

    # 3. Create the final tree
    combined_tree = ET.ElementTree(combined_root)
    
    # Optional: Adds indentation so the file is readable in text editors
    ET.indent(combined_tree, space="    ", level=0)
    
    # 4. Write to file
    with open(output_file, "wb") as f:
        combined_tree.write(f, encoding='utf-8', xml_declaration=True)
    
    print(f"Success! Created '{output_file}' with all questions.")
    
if __name__ == "__main__":
    # Ensure this path matches your local folder structure
    combine_medical_questions(input_dir="./datasets/11_MPlusDrugs_QA")

Found 1312 XML files. Processing...
Success! Created './datasets/combined_medical_questions.xml' with all questions.


In [7]:
import xml.etree.ElementTree as ET
import glob
import os
import re
import random

def transform_and_combine_xml(input_dir, output_dir, output_filename="transformed_medical_questions.xml"):
    # --- CONFIGURATION ---
    target_antibiotics = [
        "Amoxicillin", "Clarithromycin", "Azithromycin", "Doxycycline",
        "Cefuroxime", "Levofloxacin", "Cefixime", "Cefotaxime",
        "Fosfomycin", "Nitrofurantoin", "Trimethoprim-sulfamethoxazole", "Ciprofloxacin"
    ]
    
    # Mapping MedLinePlus tags to your NEW qtypes
    # Add or adjust these pairs based on how you want the original data classified
    QTYPE_MAP = {
        "side effects": "SIDE_EFFECTS",
        "important warning": "WARNING_PRECAUTIONS",
        "precautions": "WARNING_PRECAUTIONS",
        "indication": "USES_INDICATIONS",
        "usage": "USES_INDICATIONS",
        "forget a dose": "STEWARDSHIP_MISSED",
        "dietary": "FOOD_INTERACTION",
        "storage and disposal": "STEWARDSHIP_LEFTOVERS",
        "emergency or overdose": "WARNING_PRECAUTIONS",
        "other information": "ANTIBIOTIC_INFO",
        "brand names": "DICTIONARY_LOOKUP",
        "brand names of combination products": "DICTIONARY_LOOKUP"
    }

    target_lower = [a.lower() for a in target_antibiotics]
    replacement_counts = {drug: 0 for drug in target_antibiotics}
    random.seed(42)

    # Root for the new XML
    combined_root = ET.Element('CombinedMedicalQuestions')

    xml_files = sorted(glob.glob(os.path.join(input_dir, '*.xml')))
    
    for xml_file in xml_files:
        try:
            tree = ET.parse(xml_file)
            root = tree.getroot()
            original_drug = root.findtext("Focus")
            if not original_drug: continue
            
            original_drug_clean = original_drug.strip()
            original_lower = original_drug_clean.lower()

            # Skip unwanted terms
            if any(term in original_lower for term in ['marijuana', 'cannabis', 'calcium', 'vitamin']):
                continue

            # Determine Target Drug
            if original_lower in target_lower:
                chosen_target = target_antibiotics[target_lower.index(original_lower)]
            else:
                min_usage = min(replacement_counts.values())
                least_used = [d for d, count in replacement_counts.items() if count == min_usage]
                chosen_target = random.choice(least_used)
                replacement_counts[chosen_target] += 1

            # Process Questions
            for q_tag in root.findall(".//Question"):
                old_qtype = q_tag.get("qtype")
                
                # Apply the Mapping - if not found in map, we default to "ANTIBIOTIC_INFO"
                new_qtype = QTYPE_MAP.get(old_qtype, "ANTIBIOTIC_INFO")

                # Text replacement
                original_text = q_tag.text if q_tag.text else ""
                pattern = re.compile(re.escape(original_drug_clean), re.IGNORECASE)
                new_text = pattern.sub(chosen_target, original_text)

                # Create new XML element
                new_q = ET.Element("Question")
                new_q.set("qid", q_tag.get("qid", "unknown"))
                new_q.set("qtype", new_qtype) # Using the NEW type here
                new_q.text = new_text
                
                combined_root.append(new_q)
                
        except Exception as e:
            print(f"Error processing {os.path.basename(xml_file)}: {e}")

    # Save output
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    output_path = os.path.join(output_dir, output_filename)
    combined_tree = ET.ElementTree(combined_root)
    ET.indent(combined_tree, space="    ", level=0)
    
    with open(output_path, "wb") as f:
        combined_tree.write(f, encoding='utf-8', xml_declaration=True)

    print(f"Success! Saved to {output_path}")

if __name__ == "__main__":
    transform_and_combine_xml(
        input_dir="./datasets/11_MPlusDrugs_QA", 
        output_dir="./processed_output"
    )

Success! Saved to ./processed_output\transformed_medical_questions.xml


Medical Info 2019

In [12]:
pip install pandas


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
import pandas as pd
import re
import random
from collections import defaultdict

# Load dataset
med_info_raw_data = pd.read_csv("datasets/MedInfo2019-QA-Medications.csv")

# Drop unnecessary columns
med_info_raw_data.drop(['URL', 'Section Title', 'Answer'], axis=1, inplace=True)

# Antibiotics List
antibiotics = [
    "Amoxicillin",
    "Clarithromycin",
    "Azithromycin",
    "Doxycycline",
    "Cefuroxime",
    "Levofloxacin",
    "Cefixime",
    "Cefotaxime",
    "Fosfomycin",
    "Nitrofurantoin",
    "Trimethoprim-sulfamethoxazole",
    "Ciprofloxacin"
]

print(f"Original dataset size: {len(med_info_raw_data)}")
print(f"Target antibiotics: {len(antibiotics)}")

# Create a normalized version for comparison (lowercase)
target_drugs_lower = [drug.lower() for drug in antibiotics]

# Identify your columns
question_col = 'Question'  # Your question column name
drug_col = 'Focus (Drug)'  # Your drug column name

random.seed(42)

# Track counts to maintain balance
replacement_counts = defaultdict(int)

def map_to_target_drug_balanced(original_drug):
    """Map unknown drugs to maintain balanced distribution"""
    original_lower = str(original_drug).lower().strip()
    
    # Find which target drug has been used least for replacements
    if not replacement_counts:
        # If no replacements yet, initialize all counts to 0
        for drug in antibiotics:
            replacement_counts[drug] = 0
    
    min_count = min(replacement_counts.values())
    least_used = [drug for drug in antibiotics if replacement_counts[drug] == min_count]
    
    # Randomly choose among the least used
    chosen = random.choice(least_used)
    replacement_counts[chosen] += 1
    
    return chosen

# Transform the dataset
transformed_rows = []
stats = {
    'kept_original': 0,
    'replaced': 0,
    'skipped': 0,
    'skipped_terms': 0
}

for idx, row in med_info_raw_data.iterrows():
    # Get the original values
    original_question = str(row[question_col])
    original_drug = str(row[drug_col]).strip()
    
    # Skip if drug is empty
    if not original_drug or original_drug.lower() == 'nan':
        stats['skipped'] += 1
        continue
    
    # Check if the drug is already in your target list
    original_drug_lower = original_drug.lower()
    
    # Find if this drug matches any in your target list
    is_in_target = False
    matched_target = None
    
    # First, check for exact matches
    if original_drug_lower in target_drugs_lower:
        is_in_target = True
        matched_target = antibiotics[target_drugs_lower.index(original_drug_lower)]
    else:
        # Check if drug name contains any target drug or vice versa
        for target_drug, target_lower in zip(antibiotics, target_drugs_lower):
            if target_lower in original_drug_lower or original_drug_lower in target_lower:
                is_in_target = True
                matched_target = target_drug
                break
    
    if is_in_target:
        # Drug is already in your list - KEEP IT
        if original_drug != matched_target:
            # Replace with correct capitalization if needed
            pattern = re.compile(re.escape(original_drug), re.IGNORECASE)
            new_question = pattern.sub(matched_target, original_question)
            new_drug = matched_target
        else:
            # Already perfect, keep as is
            new_question = original_question
            new_drug = original_drug
        stats['kept_original'] += 1
    else:
        # Drug is NOT in your list - REPLACE IT with balanced random
        target_drug = map_to_target_drug_balanced(original_drug)
        
        # Check if drug was skipped due to skip terms
        if target_drug is None:
            stats['skipped_terms'] += 1
            continue
        
        # Replace the drug name in the question
        pattern = re.compile(re.escape(original_drug), re.IGNORECASE)
        new_question = pattern.sub(target_drug, original_question)
        new_drug = target_drug
        stats['replaced'] += 1
    
    # Create transformed row with original drug for reference
    new_row = row.to_dict()
    new_row[question_col] = new_question
    new_row[drug_col] = new_drug
    new_row['original_drug'] = original_drug  # Keep for reference
    
    transformed_rows.append(new_row)

# Create new dataframe with all transformed rows
new_df = pd.DataFrame(transformed_rows)

# Print detailed statistics
print("\n" + "="*60)
print("TRANSFORMATION STATISTICS")
print("="*60)
print(f"Total rows processed: {len(med_info_raw_data)}")
print(f"✅ Kept original drugs (already in antibiotic list): {stats['kept_original']}")
print(f"🔄 Replaced drugs (not in antibiotic list): {stats['replaced']}")
print(f"⚠️ Skipped (empty drug): {stats['skipped']}")
print(f"⚠️ Skipped (unwanted terms like marijuana, etc): {stats['skipped_terms']}")
print(f"\nFinal dataset size: {len(new_df)}")

# Show distribution of drugs in the transformed dataset
print("\n" + "="*60)
print("DRUG DISTRIBUTION IN TRANSFORMED DATASET")
print("="*60)
drug_counts = new_df[drug_col].value_counts()
for drug, count in drug_counts.items():
    percentage = (count / len(new_df)) * 100
    print(f"{drug}: {count} ({percentage:.1f}%)")

# Show how many of each original drug type
print("\n" + "="*60)
print("ORIGINAL VS TRANSFORMED SUMMARY")
print("="*60)
original_antibiotic_count = stats['kept_original']
replaced_count = stats['replaced']
print(f"Rows that originally had antibiotics: {original_antibiotic_count}")
print(f"Rows that originally had other drugs (now replaced): {replaced_count}")

# Show replacement distribution
if stats['replaced'] > 0:
    print("\n" + "="*60)
    print("REPLACEMENT DISTRIBUTION (How unknown drugs mapped to antibiotics)")
    print("="*60)
    for drug, count in replacement_counts.items():
        percentage = (count / stats['replaced']) * 100 if stats['replaced'] > 0 else 0
        print(f"{drug}: {count} ({percentage:.1f}%)")

# Show samples
print("\n" + "="*60)
print("SAMPLE TRANSFORMATIONS")
print("="*60)

# Show examples of kept drugs (originally antibiotics)
kept_samples = new_df[new_df['original_drug'] == new_df[drug_col]].head(3)
if len(kept_samples) > 0:
    print("\n✅ KEPT (drug was already an antibiotic):")
    for _, row in kept_samples.iterrows():
        print(f"  Original: {row['original_drug']} → Kept as: {row[drug_col]}")
        print(f"  Question: {row[question_col][:80]}...")
        print()

# Show examples of replaced drugs (originally non-antibiotics)
replaced_samples = new_df[new_df['original_drug'] != new_df[drug_col]].head(5)
if len(replaced_samples) > 0:
    print("\n🔄 REPLACED (drug was NOT an antibiotic):")
    for _, row in replaced_samples.iterrows():
        print(f"  Original: {row['original_drug']} → Replaced with: {row[drug_col]}")
        print(f"  Question: {row[question_col][:80]}...")
        print()

# Save transformed dataset
output_file = 'medinfo_questions.csv'
new_df.to_csv(output_file, index=False)
print(f"\n✅ Transformed dataset saved to '{output_file}'")

# Show a few original non-antibiotic drugs that got replaced
print("\n" + "="*60)
print("EXAMPLES OF ORIGINAL NON-ANTIBIOTIC DRUGS THAT WERE REPLACED")
print("="*60)
original_non_ab = new_df[new_df['original_drug'] != new_df[drug_col]]['original_drug'].value_counts().head(10)
for drug, count in original_non_ab.items():
    print(f"  {drug}: {count} occurrences")

Original dataset size: 690
Target antibiotics: 12

TRANSFORMATION STATISTICS
Total rows processed: 690
✅ Kept original drugs (already in antibiotic list): 16
🔄 Replaced drugs (not in antibiotic list): 673
⚠️ Skipped (empty drug): 1
⚠️ Skipped (unwanted terms like marijuana, etc): 0

Final dataset size: 689

DRUG DISTRIBUTION IN TRANSFORMED DATASET
Ciprofloxacin: 61 (8.9%)
Amoxicillin: 59 (8.6%)
Doxycycline: 58 (8.4%)
Azithromycin: 58 (8.4%)
Nitrofurantoin: 58 (8.4%)
Levofloxacin: 57 (8.3%)
Cefuroxime: 57 (8.3%)
Fosfomycin: 57 (8.3%)
Trimethoprim-sulfamethoxazole: 56 (8.1%)
Clarithromycin: 56 (8.1%)
Cefixime: 56 (8.1%)
Cefotaxime: 56 (8.1%)

ORIGINAL VS TRANSFORMED SUMMARY
Rows that originally had antibiotics: 16
Rows that originally had other drugs (now replaced): 673

REPLACEMENT DISTRIBUTION (How unknown drugs mapped to antibiotics)
Amoxicillin: 56 (8.3%)
Clarithromycin: 56 (8.3%)
Azithromycin: 56 (8.3%)
Doxycycline: 56 (8.3%)
Cefuroxime: 56 (8.3%)
Levofloxacin: 56 (8.3%)
Cefixime: 5

MEDQUAD

Web MD (Scraping)

In [ ]:
!pip install requests_html

In [ ]:
!pip install lxml_html_clean

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd

# Your working headers
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
}

# List of URLs to scrape
urls_to_scrape = [
    {'name': 'amoxicillin', 'url': 'https://www.webmd.com/drugs/amoxicillin-amoxil'},
    {'name': 'azithromycin', 'url': 'https://www.webmd.com/drugs/azithromycin-zithromax'},
    {'name': 'clarithromycin', 'url': 'https://www.webmd.com/drugs/clarithromycin-biaxin'},
    {'name': 'cefuroxime', 'url': 'https://www.webmd.com/drugs/cefuroxime-ceftin'},
    {'name': 'doxycycline', 'url': 'https://www.webmd.com/drugs/doxycycline-vibramycin'},
    {'name': 'levofloxacin', 'url': 'https://www.webmd.com/drugs/levofloxacin-levaquin'},
    {'name': 'cefixime', 'url': 'https://www.webmd.com/drugs/cefixime-suprax'},
    {'name': 'fosfomycin', 'url': 'https://www.webmd.com/drugs/fosfomycin-monurol'},
    {'name': 'nitrofurantoin', 'url': 'https://www.webmd.com/drugs/nitrofurantoin-macrobid-macrodantin'},  # Fixed spelling
    {'name': 'sulfamethoxazole and trimethoprim', 'url': 'https://www.webmd.com/drugs/sulfamethoxazole-trimethoprim-bactrim'},
    {'name': 'ciprofloxacin', 'url': 'https://www.webmd.com/drugs/ciprofloxacin-cipro'}
]

# Create an empty list to store all rows
all_rows = []

# Loop through each URL
for drug in urls_to_scrape:
    drug_name = drug['name']
    url = drug['url']

    print(f"\n{'='*50}")
    print(f"Scraping: {drug_name}")
    print(f"URL: {url}")
    print('='*50)

    # Make request
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # Check if page loaded correctly

    # Find headings
    titles = soup.select("div.monograph-page h3")
    print(f"Found {len(titles)} h3 elements")

    # Store each heading as a separate row
    for title in titles:
        heading_text = title.get_text(strip=True)
        if heading_text:  # Only add if there's text
            all_rows.append({
                'drug': drug_name,
                'url': url,
                'heading': heading_text
            })

    # Be polite - wait between requests
    if drug != urls_to_scrape[-1]:  # Don't wait after last item
        time.sleep(2)

# Create DataFrame from all rows
web_md_scraped_df = pd.DataFrame(all_rows)

# Save to CSV
web_md_scraped_df.to_csv("webmd_scraped.csv", index=False, encoding='utf-8')